# PPO Training on Boeing 747 Longitudinal Model

This notebook trains a Proximal Policy Optimization (PPO) agent on the improved Boeing 747 longitudinal dynamics environment. It supports both single and vectorized (64 parallel) environments with configurable reference signal types (step, sine, ramp, mixed).

In [1]:
# Optional installs (run if needed)
# %pip install tensorboard tqdm --quiet

import os
import numpy as np
import torch

from tensoraerospace.envs.b747 import ImprovedB747Env
from tensoraerospace.envs.b747_vec_torch import ImprovedB747VecEnvTorch
from tensoraerospace.agent.ppo.model import PPO

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Using device:", DEVICE)

# Reproducibility
np.random.seed(1)
_ = torch.manual_seed(1)


2026-01-02 03:33:17.006075: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-02 03:33:17.006108: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-02 03:33:17.006913: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-02 03:33:17.011765: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-02 03:33:17.704389: W tensorflow/compiler/tf2

Using device: cuda


In [2]:
# Build environment
from typing import Optional

import gymnasium as gym

from tensoraerospace.signals.standart import sinusoid_vertical_shift, unit_step
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

# Time base
dt = 0.1
_tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(_tp, dt=dt)
number_time_steps = len(_tp)

def make_reference_signal(
    tp_sec: np.ndarray,
    kind: str,
    *,
    amplitude_deg: float = 1.0,
    frequency_hz: float = 0.05,
    step_time_sec: float = 5.0,
) -> np.ndarray:
    """Return reference signal shaped as (1, T) in radians."""
    tp_sec = np.asarray(tp_sec)

    if kind == "sine":
        sig = sinusoid_vertical_shift(
            tp=tp_sec,
            frequency=frequency_hz,
            amplitude=np.deg2rad(amplitude_deg),
            vertical_shift=0.0,
        )
    elif kind == "step":
        sig = unit_step(
            tp=tp_sec,
            degree=amplitude_deg,
            time_step=step_time_sec,
            output_rad=True,
        )
    else:
        raise ValueError(f"Unknown kind={kind!r}. Use 'sine' or 'step'.")

    return np.reshape(sig, [1, -1])


class RandomReferenceSignalWrapper(gym.Wrapper):
    """Randomize reference signal on every reset.

    For robust step tracking we randomize:
    - step amplitude (deg)
    - step time (sec)

    This prevents PPO from "anticipating" a fixed step time and drifting before the step.
    """

    def __init__(
        self,
        env: gym.Env,
        tp_sec: np.ndarray,
        *,
        fixed_kind: str = "step",
        p_step: float = 0.5,
        amplitude_deg_range: tuple[float, float] = (1.0, 10.0),
        step_time_sec_range: tuple[float, float] = (1.0, 10.0),
        frequency_hz: float = 0.05,
        seed: Optional[int] = None,
    ):
        super().__init__(env)
        self.tp_sec = np.asarray(tp_sec)
        self.fixed_kind = str(fixed_kind)
        self.p_step = float(p_step)
        self.amplitude_deg_range = (float(amplitude_deg_range[0]), float(amplitude_deg_range[1]))
        self.step_time_sec_range = (float(step_time_sec_range[0]), float(step_time_sec_range[1]))
        self.frequency_hz = float(frequency_hz)
        self.rng = np.random.default_rng(seed)

        # Diagnostics
        self.last_kind: Optional[str] = None
        self.last_amplitude_deg: Optional[float] = None
        self.last_step_time_sec: Optional[float] = None

    def _sample_and_set_reference(self) -> None:
        # Choose signal kind
        kind = self.fixed_kind
        if kind == "mixed":
            kind = "step" if float(self.rng.random()) < float(self.p_step) else "sine"

        # Sample amplitude + step time
        amp_lo, amp_hi = self.amplitude_deg_range
        amp_deg = float(self.rng.uniform(amp_lo, amp_hi))

        t0 = float(self.tp_sec[0])
        t1 = float(self.tp_sec[-1])
        st_lo, st_hi = self.step_time_sec_range
        step_time_sec = float(self.rng.uniform(st_lo, st_hi))
        step_time_sec = float(np.clip(step_time_sec, t0, t1))

        self.last_kind = kind
        self.last_amplitude_deg = amp_deg
        self.last_step_time_sec = step_time_sec

        self.env.reference_signal = make_reference_signal(
            self.tp_sec,
            kind,
            amplitude_deg=amp_deg,
            frequency_hz=self.frequency_hz,
            step_time_sec=step_time_sec,
        )

    def reset(self, seed=None, options=None):
        # If user passes seed, respect it for reference sampling too.
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self._sample_and_set_reference()
        return self.env.reset(seed=seed, options=options)


# Initial state [u, w, q, theta] (SI units; angles in rad)
init_state = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

# Toggle: vectorized torch env (64 parallel envs)
USE_VEC_ENV = True
NUM_ENVS = 64

# ============================================================================
# Signal type configuration for vectorized environment
# ============================================================================
# Options:
#   - "step": Only step signals (default, best for step-response training)
#   - "sine": Only sinusoidal signals (good for tracking)
#   - "ramp": Only ramp signals (linear transitions)
#   - "mixed": Random mix of step/sine/ramp each episode (universal controller)
#
# Reward mode:
#   - "step_response": Full reward with step-specific penalties (overshoot, settling)
#   - "tracking": Universal reward for any signal (only base tracking error)
# ============================================================================
SIGNAL_TYPE = "sine"  # "step", "sine", "ramp", or "mixed"
REWARD_MODE = "tracking"  # "tracking" for universal, "step_response" for step-specific

if USE_VEC_ENV:
    # Vectorized torch env (batched) — runs in torch (CUDA if DEVICE==cuda)
    env = ImprovedB747VecEnvTorch(
        num_envs=NUM_ENVS,
        dt=dt,
        tn=float(tps[-1]),
        initial_state=init_state,
        device=DEVICE,
        seed=1,
        auto_reset=True,
        reward_mode=REWARD_MODE,  # Universal tracking or step-response specific
        step_randomization={
            # Signal type: "step", "sine", "ramp", or "mixed"
            "signal_type": SIGNAL_TYPE,
            # Amplitude range (degrees) - applies to all signal types
            "amplitude_deg_range": (-10.0, 10.0),
            "min_abs_amplitude_deg": 1.0,  # avoid near-zero trivial signals
            # Step signal parameters
            "step_time_sec_range": (2.0, 10.0),
            # Sine signal parameters
            "frequency_hz_range": (0.05, 0.01),
            # Mixed mode probabilities (p_ramp = 1 - p_step - p_sine)
            "p_step": 0.4,
            "p_sine": 0.4,
        },
    )

    obs, info = env.reset()

    # Hard guard: if you see anything other than (NUM_ENVS, 4), you're NOT using vec env
    try:
        import torch

        assert hasattr(env, "num_envs") and int(env.num_envs) == int(NUM_ENVS)
        assert torch.is_tensor(obs) and tuple(obs.shape) == (int(NUM_ENVS), 4)
    except Exception as e:
        raise RuntimeError(
            "USE_VEC_ENV=True but env/obs are not vectorized. "
            "Re-run Cell 2 top-to-bottom. Expected obs shape (NUM_ENVS, 4)."
        ) from e

    print("Vec obs shape:", tuple(obs.shape))
    print("num_envs:", env.num_envs)
    print("device:", env.device)
    print("signal_type:", SIGNAL_TYPE)
    print("reward_mode:", REWARD_MODE)
    print("Action space:", env.action_space)
    print("Observation space:", env.observation_space)
else:
    # Base env (reference gets overwritten by wrapper on reset anyway)
    base_env = ImprovedB747Env(
        initial_state=init_state,
        reference_signal=make_reference_signal(tps, "sine"),
        number_time_steps=number_time_steps,
        dt=dt,
        initial_elevator_deg=0.0,
        reward_mode=REWARD_MODE,  # Use same reward mode
    )

    # Train on randomized step times + amplitudes (robustness)
    env = RandomReferenceSignalWrapper(
        base_env,
        tps,
        fixed_kind="step",  # use "mixed" if you also want sine
        amplitude_deg_range=(1.0, 10.0),
        step_time_sec_range=(1.0, 10.0),
        frequency_hz=0.05,
        seed=1,
    )

    obs, info = env.reset()
    print("Obs shape:", np.array(obs).shape)
    print("Action space:", env.action_space)
    print("Observation space:", env.observation_space)
    print("reward_mode:", REWARD_MODE)
    print("Sampled reference kind:", env.last_kind)
    print("Sampled step amplitude (deg):", env.last_amplitude_deg)
    print("Sampled step time (sec):", env.last_step_time_sec)


Vec obs shape: (64, 4)
num_envs: 64
device: cuda
signal_type: sine
reward_mode: tracking
Action space: Box(-1.0, 1.0, (1,), float32)
Observation space: Box(-1.0, 1.0, (4,), float32)


## Environment Configuration

Build the B747 environment with randomized reference signals. The vectorized torch environment runs 64 parallel simulations on GPU for efficient data collection.

In [3]:
# Create PPO agent (tuned hyperparams)
# General tips for ImprovedB747Env:
# - Higher gamma (0.99) improves tracking stability
# - Larger rollout_len (2048) collects more diverse trajectories
# - Moderate clip_param (0.2) balances exploration and stability
# - Lower entropy_coef reduces randomness once converged

# Fast preset: make the notebook complete in minutes.
# Set FAST_PRESET=False for full training.
FAST_PRESET = True
# Use the env toggle from Cell 2 (avoid accidental mismatch)
USE_VEC_ENV = bool(globals().get("USE_VEC_ENV", True))
NUM_ENVS = int(globals().get("NUM_ENVS", 64))
# NUM_ENVS is taken from Cell 2 if defined
# NOTE:
# - If USE_VEC_ENV=True, each PPO "episode" here is one PPO update step collecting
#   rollout_len * NUM_ENVS transitions.
# - Vec env observations are already normalized to [-1, 1], so normalize_obs=False.

if USE_VEC_ENV:
    # More stable preset for vec64 (based on your TB: KL/clip_fraction were too high)
    max_episodes = 90000 if FAST_PRESET else 90000
    rollout_len = 256
    num_epochs = 10 if FAST_PRESET else 3
    batch_size = rollout_len * NUM_ENVS

    # Dedicated dirs for this run (avoid mixing old checkpoints)
    RUN_TAG = "b747_vec64_step"
    best_model_dir = f"./{RUN_TAG}_best"
    save_root_dir = f"./{RUN_TAG}"

    # Optional: wipe previous checkpoints for a clean run
    RESET_CHECKPOINT_DIRS = True
    if RESET_CHECKPOINT_DIRS:
        import shutil

        shutil.rmtree(best_model_dir, ignore_errors=True)
        shutil.rmtree(save_root_dir, ignore_errors=True)

    agent = PPO(
        env=env,
        gamma=0.99,
        max_episodes=max_episodes,
        rollout_len=rollout_len,
        clip_pram=0.2,
        num_epochs=num_epochs,
        batch_size=batch_size,
        # keep exploration so policy doesn't freeze
        entropy_coef=0.01,
        # moderate policy step size (too small => KL ~ 0, too big => instability)
        actor_lr=3e-4,
        critic_lr=1e-3,
        gae_lambda=0.95,
        max_grad_norm=0.5,
        target_kl=0.015,
        normalize_obs=False,
        normalize_reward=True,
        # widen action std range to avoid near-deterministic policy
        actor_log_std_min=-5.0,
        actor_log_std_max=-1.5,
        # tensorboard log dir (prevents mixing runs)
        log_dir=f"runs/{RUN_TAG}",
        # best checkpoint (async)
        save_best_model=True,
        best_model_dir=best_model_dir,
        save_best_async=True,
        seed=336699,
        device=DEVICE,
    )
else:
    max_episodes = 500 if FAST_PRESET else 1000
    rollout_len = 512 if FAST_PRESET else 2048
    num_epochs = 3 if FAST_PRESET else 10

    best_model_dir = "./ppo_b747_improved_best"
    save_root_dir = "./ppo_b747_improved"

    agent = PPO(
        env=env,
        gamma=0.99,
        max_episodes=max_episodes,
        rollout_len=rollout_len,
        clip_pram=0.2,
        num_epochs=num_epochs,
        batch_size=64,
        entropy_coef=0.01,
        actor_lr=3e-4,
        critic_lr=1e-3,
        gae_lambda=0.95,
        max_grad_norm=0.5,
        target_kl=0.015,
        normalize_obs=True,
        normalize_reward=False,
        # best checkpoint (async)
        save_best_model=True,
        best_model_dir=best_model_dir,
        save_best_async=True,
        seed=336699,
        device=DEVICE,
    )

print("FAST_PRESET:", FAST_PRESET)
print("USE_VEC_ENV:", USE_VEC_ENV)
print("Actor parameters:", sum(p.numel() for p in agent.actor.parameters()))
print("Critic parameters:", sum(p.numel() for p in agent.critic.parameters()))


FAST_PRESET: True
USE_VEC_ENV: True
Actor parameters: 67586
Critic parameters: 67329


## PPO Agent Configuration

Create the PPO agent with tuned hyperparameters. A fast preset is available for quick iteration; disable it for full training.

In [8]:
# Optional: load checkpoint (best if available) to continue training/evaluate
from pathlib import Path

LOAD_CHECKPOINT = True  # set True to resume training from saved checkpoint
RESET_CHECKPOINT_DIRS = True

def pick_checkpoint_dir(preferred: str, fallback_root: str) -> Path:
    preferred_p = Path(preferred)
    if preferred_p.exists() and (preferred_p / "config.json").exists():
        return preferred_p

    root = Path(fallback_root)
    if root.exists() and root.is_dir():
        # pick newest subdir that looks like a saved PPO folder
        subdirs = sorted(
            [p for p in root.iterdir() if p.is_dir()],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        for p in subdirs:
            if (p / "config.json").exists():
                return p

    raise FileNotFoundError(
        f"No checkpoint found. Tried {preferred_p.resolve()} and {root.resolve()}/*/config.json"
    )


if LOAD_CHECKPOINT:
    agent = PPO.from_pretrained('./b747_vec64_step_best_091_06st copy')
    print("Loaded agent. best_reward:", getattr(agent, "best_reward", None))
else:
    print("LOAD_CHECKPOINT=False -> using freshly created agent from Cell 3")



Loaded agent. best_reward: 0.9137129187583923


## Load Checkpoint (Optional)

Optionally resume training from a previously saved best checkpoint.

In [7]:
# Train
# Tip: set max_episodes in Cell 3 via FAST_PRESET / USE_VEC_ENV presets.

agent.train()
# Flush pending async checkpoint saves (if enabled)
try:
    agent.close()
except Exception:
    pass


  1%|          | 1098/90000 [15:32<20:58:41,  1.18it/s]


KeyboardInterrupt: 

## Train

Run the PPO training loop. Monitor progress via TensorBoard.